# Relegation battle competitiveness — Israeli Premier League

This notebook computes four relegation-oriented metrics using **only existing project files**. Results are saved to:

`notebooks/data/processed/relegation_competitiveness/`

Original data files are **not** modified.


## Missing Data Report

| Asset | Role |
|-------|------|
| `data/interim/scraped_standings/regular_final_tables/regular_final_table_*.csv` | Inventory & **Metric 3 fallback** after regular phase ONLY (legacy 2006/07–2008/09); Metric 3 primary snapshot is relegation-phase final when available |
| `data/matches/matches_*_*_ligat_haal_transfermarkt_dated.csv` (preferred) or `*_transfermarkt.csv` | Round-by-round **regular** fixtures / points rebuilt for Metric 1 |
| `data/processed/position_tables_by_round_tm/positions_by_round_*_tm.csv` | Optional legacy TM positional matrix (inventory only). **Metric 4** uses `regular_season_tracking` + `all_seasons_relegation_tracking` instead |
| `data/regular_season_tracking/regular_season_tracking_*.csv` | TM round-by-round **regular phase** standings (points, positions) |
| `data/relegation_playoff_tracking/relegation_playoff_tracking_*.csv` (or `all_seasons_relegation_tracking.csv`) | TM **relegation playoff** snapshots; **Metric 3** survival gap uses **last round** of this stage (split-era). **Metric 2** also uses these files. **Not used** for legacy 2006/07–2008/09 single-table campaigns |

**Scope note:** Metric 2 uses **concatenated regular + relegation playoff** standings **from 2009/10 onward**. Seasons **2006/07**, **2007/08**, and **2008/09** used a single **regular** fixture list (**33** rounds, no relegation playoff on TM); for those seasons Metric 2 uses **regular season tracking only** through round 33. Other metrics still follow their cell-level definitions where applicable.


In [1]:
# --- Imports and path resolution ---
from __future__ import annotations

import re
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:

    def display(obj: object, **_kwargs):
        """Notebook fallback outside Jupyter"""
        print(obj)


NOTEBOOK_ROOT = Path('.').resolve()
for _ in range(8):
    if (NOTEBOOK_ROOT / 'data').is_dir():
        break
    NOTEBOOK_ROOT = NOTEBOOK_ROOT.parent

DATA = NOTEBOOK_ROOT / 'data'
INTERIM = DATA / 'interim'
RAW = DATA / 'raw'
PROCESSED = DATA / 'processed'
MATCHES_DIR = DATA / 'matches'
REG_FINAL_DIR = INTERIM / 'scraped_standings' / 'regular_final_tables'
POSITION_TM_DIR = PROCESSED / 'position_tables_by_round_tm'

OUT_DIR = PROCESSED / 'relegation_competitiveness'
OUT_DIR.mkdir(parents=True, exist_ok=True)

METRIC1_SURVIVAL_BAND_PTS = 6  # Metric 1: contenders if abs(points - lowest_safe) <= this (rank >= N - 6)
N_REL = 2  # relegated clubs per season in this dataset

print('Data root:', DATA)
print('Output:   ', OUT_DIR)


Data root: C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data
Output:    C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\processed\relegation_competitiveness


In [2]:
def season_label(name: str) -> Optional[str]:
    """Parse regular_final_table_YYYY_YY.csv"""
    m = re.match(r'regular_final_table_(\d{4})_(\d{2})\.csv$', name, re.I)
    if not m:
        return None
    return f"{int(m.group(1))}/{m.group(2)}"


def match_file_for_season(season: str) -> Optional[Path]:
    safe = season.replace("/", "_")
    cand = [
        MATCHES_DIR / f"matches_{safe}_ligat_haal_transfermarkt_dated.csv",
        MATCHES_DIR / f"matches_{safe}_ligat_haal_transfermarkt.csv",
        RAW / f"matches_{safe}_ligat_haal_transfermarkt.csv",
    ]
    for p in cand:
        if p.is_file():
            return p
    return None


def position_tm_file(season: str) -> Optional[Path]:
    safe = season.replace("/", "_")
    p = POSITION_TM_DIR / f'positions_by_round_{safe}_tm.csv'
    return p if p.is_file() else None


def parse_score(score) -> tuple[int, int]:
    p = str(score).strip().replace(" ", "").split(":")
    return int(p[0]), int(p[1])


def load_matches(path: Path) -> pd.DataFrame:
    """Require home, away, score, round."""
    df = pd.read_csv(path)
    for c in ('home', 'away', 'score', 'round'):
        if c not in df.columns:
            raise ValueError(f'Missing {c} in {path.name}')
    df = df.dropna(subset=['home', 'away', 'score']).copy()
    df['round'] = pd.to_numeric(df['round'], errors='coerce')
    df = df.dropna(subset=['round'])
    df['round'] = df['round'].astype(int)
    return df


def total_games_per_team(matches: pd.DataFrame) -> dict[str, int]:
    from collections import Counter
    c = Counter(matches['home']) + Counter(matches['away'])
    return dict(c)


def standings_after_round(matches: pd.DataFrame, rnd: int) -> pd.DataFrame:
    sub = matches[matches['round'] <= rnd]
    teams = set(sub['home']) | set(sub['away'])
    st = {t: {'pts': 0, 'gf': 0, 'ga': 0} for t in teams}
    for _, r in sub.iterrows():
        h, a = r['home'], r['away']
        hg, ag = parse_score(r['score'])
        st[h]['gf'] += hg
        st[h]['ga'] += ag
        st[a]['gf'] += ag
        st[a]['ga'] += hg
        if hg > ag:
            st[h]['pts'] += 3
        elif hg < ag:
            st[a]['pts'] += 3
        else:
            st[h]['pts'] += 1
            st[a]['pts'] += 1
    rows = []
    for t, s in st.items():
        gd = s['gf'] - s['ga']
        rows.append({'team': t, 'points': s['pts'], 'goal_diff': gd, 'gf': s['gf']})
    return pd.DataFrame(rows).sort_values(
        ['points', 'goal_diff', 'gf'], ascending=[False, False, False]
    ).reset_index(drop=True)


def played_after_round(matches: pd.DataFrame, rnd: int) -> dict[str, int]:
    sub = matches[matches['round'] <= rnd]
    from collections import Counter
    return dict(Counter(sub['home']) + Counter(sub['away']))


def checkpoint_round(rmax: int, frac: float = 0.75) -> int:
    return max(1, int(np.ceil(frac * rmax)))


## Section A — Data inventory

Scan which seasons have regular finals, match exports, and TM position tables.


In [3]:
rows = []
for p in sorted(REG_FINAL_DIR.glob('regular_final_table_*.csv')):
    lab = season_label(p.name)
    if not lab:
        continue
    reg = pd.read_csv(p)
    mf, pf = match_file_for_season(lab), position_tm_file(lab)
    rows.append({
        'season': lab,
        'league_size': len(reg),
        'regular_final': p.name,
        'matches_file': mf.name if mf else None,
        'positions_tm_file': pf.name if pf else None,
    })
inventory_df = pd.DataFrame(rows).sort_values('season').reset_index(drop=True)
inventory_df.to_csv(OUT_DIR / 'data_inventory_relegation_metrics.csv', index=False, encoding='utf-8-sig')
display(inventory_df)


,season,league_size,regular_final,matches_file,positions_tm_file
0,2006/07,12,regular_final_table_2006_07.csv,matches_2006_07_ligat_haal_transfermarkt_dated...,positions_by_round_2006_07_tm.csv
1,2007/08,12,regular_final_table_2007_08.csv,matches_2007_08_ligat_haal_transfermarkt_dated...,positions_by_round_2007_08_tm.csv
2,2008/09,12,regular_final_table_2008_09.csv,matches_2008_09_ligat_haal_transfermarkt_dated...,positions_by_round_2008_09_tm.csv
3,2009/10,16,regular_final_table_2009_10.csv,matches_2009_10_ligat_haal_transfermarkt_dated...,positions_by_round_2009_10_tm.csv
4,2010/11,16,regular_final_table_2010_11.csv,matches_2010_11_ligat_haal_transfermarkt_dated...,positions_by_round_2010_11_tm.csv
5,2011/12,16,regular_final_table_2011_12.csv,matches_2011_12_ligat_haal_transfermarkt_dated...,positions_by_round_2011_12_tm.csv
6,2012/13,14,regular_final_table_2012_13.csv,matches_2012_13_ligat_haal_transfermarkt_dated...,positions_by_round_2012_13_tm.csv
7,2013/14,14,regular_final_table_2013_14.csv,matches_2013_14_ligat_haal_transfermarkt_dated...,positions_by_round_2013_14_tm.csv
8,2014/15,14,regular_final_table_2014_15.csv,matches_2014_15_ligat_haal_transfermarkt_dated...,positions_by_round_2014_15_tm.csv
9,2015/16,14,regular_final_table_2015_16.csv,matches_2015_16_ligat_haal_transfermarkt_dated...,positions_by_round_2015_16_tm.csv


## Metric 3 — Survival gap

For seasons **with** a relegation playoff, the survival gap is calculated based on **the final standings after the playoff**, since relegation is determined only at that stage. Only **one** final table is used (sort by **`position`** ascending, **1** = top); regular-season standings are **not** mixed into that snapshot.

Seasons **2006/07**, **2007/08**, and **2008/09** pre-date the relegation playoff on TM: gaps use **`regular_final_table`** (full ISR1 ladder).

Standings **`N`** teams, **`N_REL` = 2** relegated clubs:

- **Lowest safe** → row **`N − N_REL − 1`** (after sorting by **`position`** ascending).
- **Best relegated** → row **`N − N_REL`**.

**Gaps**: `points_gap_survival` / `gd_gap_survival` = lowest-safe minus best-relegated (points / goal difference).


In [4]:
REL_PLAYOFF_DIR = DATA / 'relegation_playoff_tracking'

# Same split-era carve-out as Metric 2: no relegation-phase table on TM
LEGACY_SINGLE_TABLE_METRIC3 = frozenset({'2006/07', '2007/08', '2008/09'})


def load_relegation_playoff_final_table(season: str) -> tuple[pd.DataFrame | None, int | None]:
    """Latest relegation_playoff Spieltag; rows sorted by ascending TM `position`.

    Prefer per-season CSV; fallback to `all_seasons_relegation_tracking.csv`.
    """

    safe = season.replace('/', '_')
    df: pd.DataFrame | None = None
    path_per = REL_PLAYOFF_DIR / f'relegation_playoff_tracking_{safe}.csv'
    if path_per.is_file():
        df = pd.read_csv(path_per)
    else:
        path_all = REL_PLAYOFF_DIR / 'all_seasons_relegation_tracking.csv'
        if path_all.is_file():
            df = pd.read_csv(path_all)
            df = df[df['season'].astype(str) == season].copy()

    if df is None or len(df) == 0:
        return None, None
    rp = df[df['stage'].astype(str).str.strip().str.lower() == 'relegation_playoff'].copy()
    if len(rp) == 0:
        return None, None
    rnd = pd.to_numeric(rp['round'], errors='coerce')
    fr = int(rnd.max())
    snap = rp[rnd == fr].sort_values(
        ['position', 'team'], ascending=[True, True],
    ).reset_index(drop=True)
    return snap, fr


m3_rows = []

for tab_path in sorted(REG_FINAL_DIR.glob('regular_final_table_*.csv')):
    season = season_label(tab_path.name)
    if not season:
        continue
    reg_preview = pd.read_csv(tab_path)
    n_pv = len(reg_preview)
    if n_pv not in (12, 14, 16):
        continue

    playoff_snap, final_rp_round = (None, None)
    if season not in LEGACY_SINGLE_TABLE_METRIC3:
        playoff_snap, final_rp_round = load_relegation_playoff_final_table(season)

    used_playoff = False

    if (
        season not in LEGACY_SINGLE_TABLE_METRIC3
        and playoff_snap is not None
        and len(playoff_snap) > N_REL
    ):
        st = playoff_snap.sort_values(
            ['position', 'team'], ascending=[True, True],
        ).reset_index(drop=True)
        used_playoff = True
    else:
        if season not in LEGACY_SINGLE_TABLE_METRIC3 and playoff_snap is None:
            print(
                f'WARN Metric3 {season}: missing relegation playoff tracking '
                '(ISR1 relegation-phase rows). Using regular_final_table fallback.'
            )
        elif season not in LEGACY_SINGLE_TABLE_METRIC3 and playoff_snap is not None and len(playoff_snap) <= N_REL:
            print(
                f'WARN Metric3 {season}: relegation_playoff snapshots exist but fewer than '
                f'{N_REL + 1} clubs at last round ({len(playoff_snap)} rows). Fallback=regular_final_table.'
            )
        reg = pd.read_csv(tab_path).sort_values('rank').reset_index(drop=True)
        st = reg.rename(columns={'rank': 'position'})

    N = len(st)
    row_ls = st.iloc[N - N_REL - 1]
    row_br = st.iloc[N - N_REL]

    pts_ls = int(row_ls['points'])
    pts_br = int(row_br['points'])
    gd_ls = int(row_ls['goal_diff'])
    gd_br = int(row_br['goal_diff'])
    tg_ls = str(row_ls['team'])
    tg_br = str(row_br['team'])

    tail = (
        st.sort_values('position', ascending=False)
        .head(min(4, N))[['team', 'points', 'goal_diff', 'position']]
        .sort_values('position')
    )
    src_txt = (
        f'relegation_playoff FINAL round={final_rp_round}' if used_playoff else 'regular_final_table fallback'
    )
    print('')
    print('=' * 64)
    print(f'Metric 3 [{season}] source={src_txt} | N={N}')
    print('Bottom four (narrowest window toward rel. slots):')
    print(tail.to_string(index=False))
    print(
        f'  Survival line: {tg_ls} pts={pts_ls} GD={gd_ls} '
        f'| Best relegated: {tg_br} pts={pts_br} GD={gd_br} '
        f'| gap_pts={pts_ls - pts_br} gap_GD={gd_ls - gd_br}'
    )

    m3_rows.append({
        'season': season,
        'league_size': N,
        'lowest_safe_team': tg_ls,
        'lowest_safe_points': pts_ls,
        'lowest_safe_goal_diff': gd_ls,
        'best_relegated_team': tg_br,
        'best_relegated_points': pts_br,
        'best_relegated_goal_diff': gd_br,
        'points_gap_survival': pts_ls - pts_br,
        'gd_gap_survival': gd_ls - gd_br,
    })

metric3_df = pd.DataFrame(m3_rows).sort_values('season')
metric3_df.to_csv(OUT_DIR / 'metric3_survival_gap.csv', index=False, encoding='utf-8-sig')
display(metric3_df)



Metric 3 [2006/07] source=regular_final_table fallback | N=12
Bottom four (narrowest window toward rel. slots):
           team  points  goal_diff  position
    Bnei Yehuda      35        -16         9
Maccabi Herzlya      34         -8        10
  Hakoah Amidar      28        -22        11
 H. Petah Tikva      16        -32        12
  Survival line: Maccabi Herzlya pts=34 GD=-8 | Best relegated: Hakoah Amidar pts=28 GD=-22 | gap_pts=6 gap_GD=14

Metric 3 [2007/08] source=regular_final_table fallback | N=12
Bottom four (narrowest window toward rel. slots):
           team  points  goal_diff  position
    Bnei Yehuda      38        -12         9
 M. Petah Tikva      37        -11        10
   H. Kfar Saba      37        -17        11
Maccabi Herzlya      30        -19        12
  Survival line: M. Petah Tikva pts=37 GD=-11 | Best relegated: H. Kfar Saba pts=37 GD=-17 | gap_pts=0 gap_GD=6

Metric 3 [2008/09] source=regular_final_table fallback | N=12
Bottom four (narrowest window towar

,season,league_size,lowest_safe_team,lowest_safe_points,lowest_safe_goal_diff,best_relegated_team,best_relegated_points,best_relegated_goal_diff,points_gap_survival,gd_gap_survival
0,2006/07,12,Maccabi Herzlya,34,-8,Hakoah Amidar,28,-22,6,14
1,2007/08,12,M. Petah Tikva,37,-11,H. Kfar Saba,37,-17,0,6
2,2008/09,12,H. Petah Tikva,31,-12,Hakoah Amidar,29,-20,2,8
3,2009/10,6,H. Ramat Gan,22,-15,Hapoel Raanana,18,-25,4,10
4,2010/11,6,H. Petah Tikva,22,-16,H. Ashkelon,19,-33,3,17
5,2011/12,8,M. Petah Tikva,40,-18,H Rishon leZion,27,-31,13,13
6,2012/13,8,Bnei Sakhnin,37,2,Maccabi Netanya,35,-5,2,7
7,2013/14,8,M. Petah Tikva,33,-18,Ramat haSharon,33,-34,0,16
8,2014/15,8,Hapoel Haifa,34,-22,H. Petah Tikva,32,-16,2,-6
9,2015/16,8,Hapoel Haifa,34,-10,Hapoel Acre,34,-21,0,11


## Metric 1 — Relegation contenders index

At round approximately **75%** of the **regular** schedule, standings are rebuilt from completed matches (**sorted by points descending**).

This metric defines relegation contenders as teams within 6 points (**above or below**) the survival line (**last safe team**), **restricted to the bottom part of the table** (`rank_snap >= N - 6` after sorting by points), so top halves are excluded.

Season output prints **`lowest_safe_points`**, the **bottom six**, the **contenders** list with gaps, optional notes on exclusions by rank, and warnings when there are **no** contenders or **only one**.


In [5]:
m1_rows = []
warn_msgs = []

for p in sorted(REG_FINAL_DIR.glob('regular_final_table_*.csv')):
    season = season_label(p.name)
    if not season:
        continue
    N = len(pd.read_csv(p))
    if N not in (12, 14, 16):
        continue
    mpath = match_file_for_season(season)
    if mpath is None:
        m1_rows.append({
            'season': season,
            'checkpoint_round': np.nan,
            'total_regular_rounds': np.nan,
            'n_teams_at_checkpoint': np.nan,
            'survival_points': np.nan,
            'min_points_among_contenders': np.nan,
            'max_points_among_contenders': np.nan,
            'n_relegation_contenders': np.nan,
            'contender_teams': None,
            'survival_gap_max_pts': METRIC1_SURVIVAL_BAND_PTS,
            'note': 'No match file',
        })
        continue
    m = load_matches(mpath)
    rmax = int(m['round'].max())
    chk = checkpoint_round(rmax, 0.75)
    st = standings_after_round(m, chk).reset_index(drop=True)
    st['rank_snap'] = np.arange(1, len(st) + 1)

    sep = '=' * 72
    print(sep + '\n[M1] Season ' + str(season) + ' | N = ' + str(N) + ' | checkpoint round = ' + str(chk) + ' / ' + str(rmax))

    if len(st) != N:
        warn = (
            f'Season {season}: Checkpoint standings have {len(st)} teams '
            f'but regular final lists {N}; skip Metric 1 for this season.'
        )
        print(warn)
        warn_msgs.append(warn)
        m1_rows.append({
            'season': season,
            'checkpoint_round': chk,
            'total_regular_rounds': rmax,
            'n_teams_at_checkpoint': len(st),
            'survival_points': np.nan,
            'min_points_among_contenders': np.nan,
            'max_points_among_contenders': np.nan,
            'n_relegation_contenders': np.nan,
            'contender_teams': None,
            'survival_gap_max_pts': METRIC1_SURVIVAL_BAND_PTS,
            'note': warn,
        })
        continue

    lowest_safe_idx = N - N_REL - 1
    lowest_safe_points = int(st.iloc[lowest_safe_idx]['points'])

    st_ev = st.copy()
    st_ev['gap_to_survival'] = (st_ev['points'] - lowest_safe_points).abs()
    cand = st_ev[st_ev['gap_to_survival'] <= METRIC1_SURVIVAL_BAND_PTS].copy()

    rank_floor = N - 6
    cont = cand[cand['rank_snap'] >= rank_floor].copy()

    if len(cont) and (cont['rank_snap'] < rank_floor).any():
        msg = (
            'ERROR: contender(s) with rank_snap < '
            + str(rank_floor)
            + ' (must not appear above cutoff)'
        )
        print(msg)
        warn_msgs.append(season + ': ' + msg)

    print(f'lowest_safe_points (lowest_safe_idx={lowest_safe_idx}): {lowest_safe_points}')
    print(
        f'Band: abs(points - lowest_safe_points) <= {METRIC1_SURVIVAL_BAND_PTS}; '
        f'rank_snap >= N - 6 = {rank_floor}'
    )

    excluded_by_rank = cand[cand['rank_snap'] < rank_floor]
    if len(excluded_by_rank):
        print(
            'Within band but excluded (rank above bottom block '
            + f'(rank_snap < {rank_floor})):'
        )
        print(
            excluded_by_rank[
                ['rank_snap', 'team', 'points', 'gap_to_survival']
            ].to_string(index=False)
        )

    bot6 = st.tail(6)[['rank_snap', 'team', 'points']].copy()
    print('Full bottom 6 teams at checkpoint:')
    print(bot6.to_string(index=False))

    mn_c = float(cont['points'].min()) if len(cont) else float('nan')
    mx_c = float(cont['points'].max()) if len(cont) else float('nan')

    note_parts = ['Regular-season snapshot from matches']

    print('Contenders list:')
    if len(cont):
        print(cont[['rank_snap', 'team', 'points', 'gap_to_survival']].to_string(index=False))
        print(f'min contender pts = {mn_c}, max contender pts = {mx_c}')
    else:
        print('(none)')
        print('Warning: No relegation battle')
        note_parts.append('No relegation battle')

    if len(cont) == 1:
        print('Warning: only 1 contender — likely incorrect (verify snapshot)')
        note_parts.append('Only one contender — likely incorrect')

    m1_rows.append({
        'season': season,
        'checkpoint_round': chk,
        'total_regular_rounds': rmax,
        'n_teams_at_checkpoint': N,
        'survival_points': lowest_safe_points,
        'min_points_among_contenders': mn_c,
        'max_points_among_contenders': mx_c,
        'n_relegation_contenders': len(cont),
        'contender_teams': '; '.join(cont['team'].astype(str)) if len(cont) else '',
        'survival_gap_max_pts': METRIC1_SURVIVAL_BAND_PTS,
        'note': ' — '.join(note_parts),
    })
    print(sep)

if warn_msgs:
    print('\n'.join(warn_msgs))

metric1_df = pd.DataFrame(m1_rows).sort_values('season')
metric1_df.to_csv(OUT_DIR / 'metric1_relegation_contenders_index.csv', index=False, encoding='utf-8-sig')
display(metric1_df)


[M1] Season 2006/07 | N = 12 | checkpoint round = 25 / 33
lowest_safe_points (lowest_safe_idx=9): 22
Band: abs(points - lowest_safe_points) <= 6; rank_snap >= N - 6 = 6
Full bottom 6 teams at checkpoint:
 rank_snap            team  points
         7   Maccabi Haifa      35
         8    H. Kfar Saba      32
         9     Bnei Yehuda      27
        10   Hakoah Amidar      22
        11 Maccabi Herzlya      21
        12  H. Petah Tikva      14
Contenders list:
 rank_snap            team  points  gap_to_survival
         9     Bnei Yehuda      27                5
        10   Hakoah Amidar      22                0
        11 Maccabi Herzlya      21                1
min contender pts = 21.0, max contender pts = 27.0
[M1] Season 2007/08 | N = 12 | checkpoint round = 25 / 33
lowest_safe_points (lowest_safe_idx=9): 26
Band: abs(points - lowest_safe_points) <= 6; rank_snap >= N - 6 = 6
Full bottom 6 teams at checkpoint:
 rank_snap            team  points
         7    H. Kfar Saba      30
 

,season,checkpoint_round,total_regular_rounds,n_teams_at_checkpoint,survival_points,min_points_among_contenders,max_points_among_contenders,n_relegation_contenders,contender_teams,survival_gap_max_pts,note
0,2006/07,25,33,12,22,21.0,27.0,3,Bnei Yehuda; Hakoah Amidar; Maccabi Herzlya,6,Regular-season snapshot from matches
1,2007/08,25,33,12,26,24.0,30.0,6,M. Petah Tikva; H. Kfar Saba; M. Tel Aviv; Hap...,6,Regular-season snapshot from matches
2,2008/09,25,33,12,23,18.0,27.0,4,H. Petah Tikva; Bnei Sakhnin; Kiryat Shmona; H...,6,Regular-season snapshot from matches
3,2009/10,23,30,16,17,17.0,22.0,5,H. Petah Tikva; H. Ramat Gan; Hapoel Acre; Hap...,6,Regular-season snapshot from matches
4,2010/11,23,30,16,20,18.0,25.0,4,B. Jerusalem; H. Petah Tikva; H. Ashkelon; Bne...,6,Regular-season snapshot from matches
5,2011/12,23,30,16,21,19.0,26.0,6,B. Jerusalem; M. Petah Tikva; H. Beer Sheva; H...,6,Regular-season snapshot from matches
6,2012/13,20,26,14,19,15.0,21.0,5,Maccabi Netanya; Bnei Sakhnin; H. Ramat Gan; H...,6,Regular-season snapshot from matches
7,2013/14,20,26,14,16,13.0,20.0,5,Hapoel Raanana; Hapoel Acre; M. Petah Tikva; R...,6,Regular-season snapshot from matches
8,2014/15,20,26,14,22,18.0,24.0,6,FC Ashdod; Hapoel Tel Aviv; Maccabi Haifa; Bne...,6,Regular-season snapshot from matches
9,2015/16,20,26,14,21,18.0,26.0,6,Bnei Yehuda; H. Kfar Saba; Bnei Sakhnin; Hapoe...,6,Regular-season snapshot from matches


## Metric 2 — Relegation decision timing (mathematical survival line)

This metric determines the exact round in which relegation became **mathematically unavoidable**, based on **actual** standings after each round and **remaining matches**, including the **relegation playoff** stage.

Standings source: **regular season tracking** concatenated with **relegation playoff tracking** for the same season (sorted by continuous `round`). It does **not** use approximate max-points sorting across every club at once.

At each round `r`, take the standings table sorted by ascending `position`. Let `N` be the row count after that round (`N` differs between the full-league phase and the relegation mini-table). Lowest safe rank corresponds to survival index `(N − N_REL − 1)` in zero-based row order (`pts_survival` = points of that club). Elimination triggers at the first round where `points_team + 3 * (total_rounds - r)` is **strictly less than** `pts_survival`. **Tie on points** with the survival-line club does not trigger that condition; if the club is still in the bottom `N_REL` on the **final** table (e.g. settled on goal difference), elimination is recorded at **`total_rounds`** with an explicit tie-break note.

`total_rounds`: **2006/07**, **2007/08**, and **2008/09** are single-table ISR1 seasons (**33** TM Spieltage, no relegation playoff). From **2009/10** onward it equals the combined regular + relegation-playoff TM `round` ceiling. Seasons that omit the relegation-playoff CSV purely because of TM gaps (not legacy) are skipped with a notebook warning.

**Outputs (per relegated club):** `season`, `team`, `elimination_round`, `total_rounds`, `rounds_before_end`. If neither condition fires before the end, `elimination_round` is NaN (“still mathematically alive” through the last snapshot).


In [6]:
# Metric 2: survival-line elimination using TM regular + relegation playoff tracking

REG_TRACK_DIR = DATA / 'regular_season_tracking'
REL_PLAYOFF_DIR = DATA / 'relegation_playoff_tracking'

# Pre-split Ligat era: full season is one regular-phase table (typically 33 matchdays TM); no relegation playoff file
LEGACY_SINGLE_TABLE_SEASONS = frozenset({'2006/07', '2007/08', '2008/09'})
LEGACY_TOTAL_ROUNDS = 33


def _standings_after_round_tracking(full_df: pd.DataFrame, rnd: int) -> pd.DataFrame:
    rows = full_df[full_df['round'] == rnd].copy()
    if rows.empty:
        return pd.DataFrame()
    return rows.sort_values(
        ['position', 'goal_diff', 'team'], ascending=[True, False, True]
    ).reset_index(drop=True)


def _compute_metric2_team(
    full_df: pd.DataFrame,
    team: str,
    total_rounds: int,
) -> tuple[int | None, float, float, int, bool]:
    """First round r where points-only max < survival line; else last round if GD / tie-break.

    When points stay tied with the lowest safe team, `max_possible < pts_survival` never
    holds, but relegation can still be decided on goal difference on the last day — then
    we attribute elimination to the final round.
    """
    elimination_r: int | None = None
    pts_at = np.nan
    surv_at = np.nan
    rem_at = 0
    for r in range(1, total_rounds + 1):
        st = _standings_after_round_tracking(full_df, r)
        if st.empty:
            continue
        if team not in set(st['team'].astype(str)):
            continue
        n_table = len(st)
        if n_table <= N_REL:
            print(
                f'WARN round {r}: table size {n_table}; cannot place survival line.'
            )
            continue
        survival_idx = n_table - N_REL - 1
        pts_team = float(st.loc[st['team'].astype(str) == team].iloc[0]['points'])
        pts_survival = float(st.iloc[survival_idx]['points'])
        remaining_games = total_rounds - r
        max_possible = pts_team + 3 * remaining_games
        if max_possible < pts_survival:
            elimination_r = r
            pts_at = pts_team
            surv_at = pts_survival
            rem_at = remaining_games
            break

    used_tiebreak_last_round = False
    if elimination_r is None:
        st = _standings_after_round_tracking(full_df, total_rounds)
        if not st.empty and team in set(st['team'].astype(str)):
            n_table = len(st)
            if n_table > N_REL:
                survival_idx = n_table - N_REL - 1
                row_team = st[st['team'].astype(str) == team].iloc[0]
                pos = int(float(row_team['position']))
                cutoff_rank = n_table - N_REL + 1  # relegation zone: worst N_REL ranks
                if pos >= cutoff_rank:
                    elimination_r = total_rounds
                    pts_at = float(row_team['points'])
                    surv_at = float(st.iloc[survival_idx]['points'])
                    rem_at = 0
                    used_tiebreak_last_round = True

    return elimination_r, pts_at, surv_at, rem_at, used_tiebreak_last_round


m2_rows = []

for table_path in sorted(REG_FINAL_DIR.glob('regular_final_table_*.csv')):
    season = season_label(table_path.name)
    if not season:
        continue
    reg_final_preview = pd.read_csv(table_path)
    n_preview = len(reg_final_preview)
    if n_preview not in (12, 14, 16):
        continue

    safe = season.replace('/', '_')
    path_reg_track = REG_TRACK_DIR / f'regular_season_tracking_{safe}.csv'
    path_rel_po = REL_PLAYOFF_DIR / f'relegation_playoff_tracking_{safe}.csv'

    if not path_reg_track.is_file():
        print(f'WARN {season}: no regular season tracking CSV: {path_reg_track}')
        continue

    reg_track = pd.read_csv(path_reg_track)

    playoff_missing = not path_rel_po.is_file()
    legacy_regular_only = playoff_missing and (season in LEGACY_SINGLE_TABLE_SEASONS)

    if playoff_missing and not legacy_regular_only:
        print(
            f'MISSING RELEGATION PLAYOFF DATA ({season}). '
            f'Expected {path_rel_po.name}; Metric 2 skipped for this season.'
        )
        continue

    if legacy_regular_only:
        rnum = pd.to_numeric(reg_track['round'], errors='coerce')
        full_tm = reg_track[rnum <= LEGACY_TOTAL_ROUNDS].copy()
        rmx_series = pd.to_numeric(full_tm['round'], errors='coerce').dropna()
        if rmx_series.empty:
            print(f'WARN {season}: empty regular tracking after filtering; skipped')
            continue
        data_max_round = int(rmx_series.max())
        if data_max_round < LEGACY_TOTAL_ROUNDS:
            print(
                f'WARN {season}: tracker max round {data_max_round} < {LEGACY_TOTAL_ROUNDS} '
                '(incomplete RR on file)'
            )
        total_rounds = max(1, min(LEGACY_TOTAL_ROUNDS, data_max_round))
    else:
        rel_track = pd.read_csv(path_rel_po)
        full_tm = pd.concat([reg_track, rel_track], ignore_index=True)
        rounds_present = sorted(full_tm['round'].dropna().unique().astype(int).tolist())
        total_rounds = int(rounds_present[-1])
    last_snap = (
        full_tm[full_tm['round'] == total_rounds]
        .sort_values(['position', 'goal_diff'], ascending=[True, False])
        .reset_index(drop=True)
    )
    final_relegated = last_snap.iloc[-N_REL:]['team'].astype(str).tolist()

    print('\n', '=' * 56)
    _hdr = (
        f'legacy {LEGACY_TOTAL_ROUNDS}-round regular TM (no relegation playoff)'
        if legacy_regular_only else
        'regular + relegation playoff'
    )
    print(f'{season}: total_rounds={total_rounds} ({_hdr}), N@{total_rounds}={len(last_snap)}')
    print(f'  Final relegated clubs (bottom {N_REL}, last TM snapshot): {final_relegated}')

    for tm in final_relegated:
        elimination_r, pts_at, surv_at, rem_at, gd_last = _compute_metric2_team(
            full_tm, tm, total_rounds
        )
        rbe = (
            np.nan if elimination_r is None else float(total_rounds - elimination_r)
        )

        warns: list[str] = []
        if legacy_regular_only:
            warns.append(
                'Legacy single-table season: relegation playoff N/A '
                f'({total_rounds} rounds ISR1 TM regular)'
            )
        if gd_last:
            warns.append(
                'Final round: relegation on tie-breaker (e.g. goal difference); '
                'equal (or non-strict) points vs survival line so no earlier points-only date'
            )
        elif elimination_r is None:
            warns.append(
                'elimination_round=None: not in relegation zone on final table (unexpected)'
            )
        if elimination_r is not None:
            print(
                f'  [{tm}] elim_round={elimination_r} | pts_team={pts_at:g} | '
                f'pts_survival={surv_at:g} | remaining_games={rem_at}'
                + (' | tie-break path' if gd_last else '')
            )
            if elimination_r < 10:
                warns.append(f'WARN: elimination very early (round={elimination_r})')
        else:
            print(
                f'  [{tm}] elimination_round=None: no final relegation snapshot match'
            )

        m2_rows.append({
            'season': season,
            'team': tm,
            'elimination_round': np.nan if elimination_r is None else float(elimination_r),
            'total_rounds': float(total_rounds),
            'rounds_before_end': rbe,
            'note': '; '.join(warns),
        })

metric2_df = pd.DataFrame(m2_rows).sort_values(['season', 'team'])
metric2_df.to_csv(OUT_DIR / 'metric2_relegation_decision_timing.csv', index=False, encoding='utf-8-sig')
display(metric2_df)



2006/07: total_rounds=33 (legacy 33-round regular TM (no relegation playoff)), N@33=12
  Final relegated clubs (bottom 2, last TM snapshot): ['Hakoah Amidar', 'H. Petah Tikva']
  [Hakoah Amidar] elim_round=32 | pts_team=27 | pts_survival=34 | remaining_games=1
  [H. Petah Tikva] elim_round=29 | pts_team=14 | pts_survival=31 | remaining_games=4

2007/08: total_rounds=33 (legacy 33-round regular TM (no relegation playoff)), N@33=12
  Final relegated clubs (bottom 2, last TM snapshot): ['H. Kfar Saba', 'Maccabi Herzlya']
  [H. Kfar Saba] elim_round=33 | pts_team=37 | pts_survival=37 | remaining_games=0 | tie-break path
  [Maccabi Herzlya] elim_round=32 | pts_team=30 | pts_survival=34 | remaining_games=1

2008/09: total_rounds=33 (legacy 33-round regular TM (no relegation playoff)), N@33=12
  Final relegated clubs (bottom 2, last TM snapshot): ['Hakoah Amidar', 'Kiryat Shmona']
  [Hakoah Amidar] elim_round=32 | pts_team=26 | pts_survival=30 | remaining_games=1
  [Kiryat Shmona] elim_round

,season,team,elimination_round,total_rounds,rounds_before_end,note
1,2006/07,H. Petah Tikva,29.0,33.0,4.0,Legacy single-table season: relegation playoff...
0,2006/07,Hakoah Amidar,32.0,33.0,1.0,Legacy single-table season: relegation playoff...
2,2007/08,H. Kfar Saba,33.0,33.0,0.0,Legacy single-table season: relegation playoff...
3,2007/08,Maccabi Herzlya,32.0,33.0,1.0,Legacy single-table season: relegation playoff...
4,2008/09,Hakoah Amidar,32.0,33.0,1.0,Legacy single-table season: relegation playoff...
5,2008/09,Kiryat Shmona,33.0,33.0,0.0,Legacy single-table season: relegation playoff...
6,2009/10,Hapoel Raanana,35.0,35.0,0.0,
7,2009/10,M. Ahi Nazareth,35.0,35.0,0.0,
8,2010/11,H. Ashkelon,35.0,35.0,0.0,
9,2010/11,H. Ramat Gan,32.0,35.0,3.0,


## Metric 4 — Relegation-zone volatility

This metric measures how frequently teams entered and exited the relegation zone across the **entire season**, **including the relegation playoff stage**, using TM **`position`** snapshots on **one continuous `round`** axis (regular **plus** `all_seasons_relegation_tracking.csv`; no `positions_by_round_*` file).

Each round snapshot has **`N`** clubs listed. Clubs in zone have **`position >= N - N_REL + 1`** with **`N_REL` = 2**. Change counts use half the symmetric difference between successive rounds; **`avg_changes_per_round`** divides by **`#rounds - 1`**.


In [7]:
# Metric 4 — relegation-zone volatility (unified TM tracking: regular + relegation playoff)

REG_TRACK_DIR_M4 = DATA / 'regular_season_tracking'
REL_PLAYOFF_DIR_M4 = DATA / 'relegation_playoff_tracking'
REL_ALL_AGG = REL_PLAYOFF_DIR_M4 / 'all_seasons_relegation_tracking.csv'

# No ISR1 relegation-phase block in legacy single-table eras
LEGACY_SINGLE_TABLE_METRIC4 = frozenset({'2006/07', '2007/08', '2008/09'})


def load_metric4_unified_tracking() -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    for fp in sorted(REG_TRACK_DIR_M4.glob('regular_season_tracking_*.csv')):
        low = fp.name.lower()
        if 'bak' in low or fp.name.startswith('all_seasons'):
            continue
        parts.append(pd.read_csv(fp))
    if not parts:
        raise FileNotFoundError(f'No regular season tracking CSV files under {REG_TRACK_DIR_M4}')
    reg = pd.concat(parts, ignore_index=True)
    if REL_ALL_AGG.is_file():
        rel = pd.read_csv(REL_ALL_AGG)
        return pd.concat([reg, rel], ignore_index=True)
    print(f'WARN Metric4 load: missing {REL_ALL_AGG.name}; using regular ISR1 tracking only.')
    return reg


FULL_M4 = load_metric4_unified_tracking()


def relegation_zone_teams_snapshot(st: pd.DataFrame, n_rel: int) -> set[str]:
    st = st.drop_duplicates(subset=['team']).copy()
    pos_num = pd.to_numeric(st['position'], errors='coerce')
    n_tab = len(st)
    if n_tab <= n_rel:
        return set()
    cut = n_tab - n_rel + 1
    return set(st.loc[pos_num >= cut, 'team'].astype(str))


m4_rows = []

for tab_path in sorted(REG_FINAL_DIR.glob('regular_final_table_*.csv')):
    season = season_label(tab_path.name)
    if not season:
        continue
    n_pv = len(pd.read_csv(tab_path))
    if n_pv not in (12, 14, 16):
        continue

    df_s = FULL_M4[FULL_M4['season'].astype(str) == season].copy()
    df_s['_rnd'] = pd.to_numeric(df_s['round'], errors='coerce')
    df_s = df_s[np.isfinite(df_s['_rnd'])].sort_values(
        by=['_rnd', 'stage'], ascending=[True, True], kind='mergesort',
    )

    if len(df_s) == 0:
        print(f'WARN M4 {season}: no unified tracking rows; skipped.')
        continue

    rnd_list = sorted(df_s['_rnd'].unique())
    playoffs_present = df_s['stage'].astype(str).str.strip().str.lower().eq(
        'relegation_playoff',
    ).any()

    if season not in LEGACY_SINGLE_TABLE_METRIC4 and not playoffs_present:
        print(
            f'WARN M4 {season}: expected relegation_playoff snapshots for split-era; '
            f'zones use regular-only standings.'
        )

    zone_sets: list[set[str]] = []

    for r in rnd_list:
        blk = df_s[df_s['_rnd'] == r]
        zone_sets.append(relegation_zone_teams_snapshot(blk, N_REL))

    chg = 0
    for i in range(len(zone_sets) - 1):
        chg += len(zone_sets[i].symmetric_difference(zone_sets[i + 1])) // 2

    union_tm = set()
    for zs in zone_sets:
        union_tm |= zs

    n_iv = len(zone_sets) - 1
    avg_chg = (chg / n_iv) if n_iv > 0 else np.nan

    if len(rnd_list) < 5:
        print(
            f'WARN M4 {season}: trajectory only has {len(rnd_list)} rounds — averages noisy.'
        )

    def show_z(seq: list[set[str]]) -> str:
        out = []
        for sset in seq:
            out.append('[' + ','.join(sorted(sset)) + ']')
        return ' | '.join(out)

    zf = zone_sets[: min(3, len(zone_sets))]
    zl = zone_sets[max(0, len(zone_sets) - 3) :]

    print('')
    print('Metric4', season, f'playoff_in_mix={playoffs_present}')
    print('  rounds=', len(rnd_list), '| half_symmetric_changes=', chg)
    print('  first zones:', show_z(zf))
    print('  last zones:', show_z(zl))

    m4_rows.append({
        'season': season,   
        'total_relegation_zone_changes': int(chg),
        'unique_teams_ever_in_zone': len(union_tm),
        'avg_changes_per_round': float(avg_chg) if avg_chg == avg_chg else np.nan,
    })

metric4_df = pd.DataFrame(m4_rows).sort_values('season')
metric4_df.to_csv(OUT_DIR / 'metric4_relegation_zone_volatility.csv', index=False, encoding='utf-8-sig')
display(metric4_df)



Metric4 2006/07 playoff_in_mix=False
  rounds= 33 | half_symmetric_changes= 8
  first zones: [H. Petah Tikva,M. Tel Aviv] | [H. Petah Tikva,M. Tel Aviv] | [H. Petah Tikva,M. Tel Aviv]
  last zones: [H. Petah Tikva,Hakoah Amidar] | [H. Petah Tikva,Hakoah Amidar] | [H. Petah Tikva,Hakoah Amidar]

Metric4 2007/08 playoff_in_mix=False
  rounds= 33 | half_symmetric_changes= 10
  first zones: [Bnei Yehuda,Maccabi Herzlya] | [Maccabi Haifa,Maccabi Herzlya] | [Maccabi Haifa,Maccabi Herzlya]
  last zones: [FC Ashdod,Maccabi Herzlya] | [H. Kfar Saba,Maccabi Herzlya] | [H. Kfar Saba,Maccabi Herzlya]

Metric4 2008/09 playoff_in_mix=False
  rounds= 33 | half_symmetric_changes= 13
  first zones: [B. Jerusalem,FC Ashdod] | [B. Jerusalem,Kiryat Shmona] | [Bnei Sakhnin,Bnei Yehuda]
  last zones: [Hakoah Amidar,Kiryat Shmona] | [Hakoah Amidar,Kiryat Shmona] | [Hakoah Amidar,Kiryat Shmona]

Metric4 2009/10 playoff_in_mix=True
  rounds= 35 | half_symmetric_changes= 12
  first zones: [Bnei Sakhnin,FC Ashd

,season,total_relegation_zone_changes,unique_teams_ever_in_zone,avg_changes_per_round
0,2006/07,8,4,0.250000
1,2007/08,10,7,0.312500
2,2008/09,13,6,0.406250
3,2009/10,12,7,0.352941
4,2010/11,11,6,0.323529
5,2011/12,5,6,0.138889
6,2012/13,12,7,0.375000
7,2013/14,12,4,0.375000
8,2014/15,17,7,0.531250
9,2015/16,13,8,0.406250


## Combined summary and charts

One row per season where joins exist. Metric 2 is summarized from **team-level** survival-line rows (`min` / `max` `elimination_round` across the relegated clubs present in that season).


In [8]:
# --- Aggregate metric 2 to season-level ---
m2s = metric2_df.dropna(subset=['elimination_round']).groupby('season', as_index=False).agg(
    first_confirmation_round=('elimination_round', 'min'),
    last_confirmation_round=('elimination_round', 'max'),
    mean_rounds_before_end=('rounds_before_end', 'mean'),
)

summary = (
    metric1_df[['season', 'n_relegation_contenders', 'checkpoint_round']]
    .merge(metric3_df[['season', 'points_gap_survival', 'gd_gap_survival']], on='season', how='outer')
    .merge(metric4_df[['season', 'total_relegation_zone_changes', 'unique_teams_ever_in_zone', 'avg_changes_per_round']], on='season', how='outer')
    .merge(m2s, on='season', how='left')
)
summary.to_csv(OUT_DIR / 'summary_all_metrics_by_season.csv', index=False, encoding='utf-8-sig')
display(summary)

# --- Plots ---
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
s = summary.dropna(subset=['n_relegation_contenders'])
axes[0, 0].plot(s['season'], s['n_relegation_contenders'], marker='o')
axes[0, 0].set_title('Metric 1: relegation contenders at ~75%')
axes[0, 0].tick_params(axis='x', rotation=75)

s3 = summary.dropna(subset=['points_gap_survival'])
axes[0, 1].plot(s3['season'], s3['points_gap_survival'], marker='o', color='darkgreen')
axes[0, 1].set_title('Metric 3: survival points gap')
axes[0, 1].tick_params(axis='x', rotation=75)

s4 = summary.dropna(subset=['total_relegation_zone_changes'])
axes[1, 0].bar(s4['season'], s4['total_relegation_zone_changes'])
axes[1, 0].set_title('Metric 4: zone membership changes')
axes[1, 0].tick_params(axis='x', rotation=75)

s2 = summary.dropna(subset=['mean_rounds_before_end'])
axes[1, 1].plot(s2['season'], s2['mean_rounds_before_end'], marker='o', color='purple')
axes[1, 1].set_title('Metric 2: mean rounds before end (avg. rel. teams)')
axes[1, 1].tick_params(axis='x', rotation=75)

plt.tight_layout()
plt.savefig(OUT_DIR / 'relegation_metrics_charts.png', dpi=120, bbox_inches='tight')
plt.close()


,season,n_relegation_contenders,checkpoint_round,points_gap_survival,gd_gap_survival,total_relegation_zone_changes,unique_teams_ever_in_zone,avg_changes_per_round,first_confirmation_round,last_confirmation_round,mean_rounds_before_end
0,2006/07,3,25,6,14,8,4,0.250000,29.0,32.0,2.5
1,2007/08,6,25,0,6,10,7,0.312500,32.0,33.0,0.5
2,2008/09,4,25,2,8,13,6,0.406250,32.0,33.0,0.5
3,2009/10,5,23,4,10,12,7,0.352941,35.0,35.0,0.0
4,2010/11,4,23,3,17,11,6,0.323529,32.0,35.0,1.5
5,2011/12,6,23,13,13,5,6,0.138889,34.0,35.0,2.5
6,2012/13,5,20,2,7,12,7,0.375000,32.0,33.0,0.5
7,2013/14,5,20,0,16,12,4,0.375000,33.0,33.0,0.0
8,2014/15,6,20,2,-6,17,7,0.531250,32.0,33.0,0.5
9,2015/16,6,20,0,11,13,8,0.406250,28.0,33.0,2.5


## Research questions

1. Has the relegation battle in the Israeli Premier League become more or less competitive over time?
2. In which seasons was the relegation battle decided closest to the final round (under the max-points elimination model)?
3. How many teams were realistically involved in the relegation battle at the ~75% checkpoint each season?
4. Does a smaller final survival gap indicate a more competitive season?
5. Are seasons with more relegation-zone changes also perceived as more competitive?
6. Which metric best reflects relegation-battle competitiveness?
7. How can relegation battle competitiveness be combined with other league competitiveness metrics in the project?


הוספת ניתוחים על העונה הנוכחית 

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

FILE_NAME = "matches_2025_26_ligat_haal_championship_round_transfermarkt.csv"

def find_input_file(filename: str) -> Path:
    candidates = [
        Path.cwd() / "notebooks" / "data" / "playoffs" / "scraped_betexplorer" / filename,
        Path.cwd() / "data" / "playoffs" / "scraped_betexplorer" / filename,
        Path.cwd() / "ligat_haal_project" / "notebooks" / "data" / "playoffs" / "scraped_betexplorer" / filename,
        Path.cwd() / "ligat_haal_project" / "data" / "playoffs" / "scraped_betexplorer" / filename,
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Could not find {filename} in any expected playoffs/scraped_betexplorer folder")


FILE_PATH = find_input_file(FILE_NAME)
print(f"Loading: {FILE_PATH}")

df = pd.read_csv(FILE_PATH)

df["HomeGoals"] = pd.to_numeric(df["HomeGoals"])
df["AwayGoals"] = pd.to_numeric(df["AwayGoals"])
df = df.sort_values(["Matchday"])

teams = sorted(
    set(df["Home"].unique()).union(df["Away"].unique())
)

points = {team: 0 for team in teams}

tracking = []

for md in sorted(df["Round"].unique()):

    round_matches = df[df["Round"] == md]

    for _, row in round_matches.iterrows():

        home = row["Home"]
        away = row["Away"]

        hg = row["HomeGoals"]
        ag = row["AwayGoals"]

        if hg > ag:
            points[home] += 3

        elif hg < ag:
            points[away] += 3

        else:
            points[home] += 1
            points[away] += 1

    table = (
        pd.DataFrame({
            "Team": list(points.keys()),
            "Points": list(points.values())
        })
        .sort_values("Points", ascending=False)
        .reset_index(drop=True)
    )

    table["Position"] = table.index + 1
    table["Round"] = md

    tracking.append(table)

standings_df = pd.concat(tracking, ignore_index=True)

standings_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'ligat_haal_project\\notebooks\\data\\playoffs\\scraped_betexplorer\\matches_2025_26_ligat_haal_championship_round_transfermarkt.csv'